In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import nltk
from nltk.corpus import stopwords

from sklearn.svm import SVC
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [15]:
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/dinislam/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [16]:
df = pd.read_csv('films.csv')
df = df['description']
df

0      Редакция Кинопоиска разбирается, как устроена ...
1      Тяжело переживая трагедию, из‑за которой ей пр...
2      В жизни Дениса Кукояки хватало съемок, приключ...
3      На Землю нападают пришельцы из космоса. Просто...
4      Семья поручила Минато Кусуноки присматривать з...
                             ...                        
656    Удивительное путешествие по Франции маленького...
657    Чунцин, начало 2000-х годов. Во время поездки ...
658    Мультипликационный фильм по произведениям Бори...
659    Смита живет в Индии и мечтает дать своей мален...
660    Единственное занятие Ленни – бесконечные вечер...
Name: description, Length: 661, dtype: object

In [17]:
df.info()

<class 'pandas.core.series.Series'>
RangeIndex: 661 entries, 0 to 660
Series name: description
Non-Null Count  Dtype 
--------------  ----- 
661 non-null    object
dtypes: object(1)
memory usage: 5.3+ KB


In [18]:
russian_stopwords = stopwords.words("russian") 
russian_stopwords.extend(['т.д.', 'т', 'д', 'это','который','свой','своём','всем','всё','её','оба','ещё','должный']) 

In [20]:
# TF-IDF векторизация
vectorizer = TfidfVectorizer(
    max_features=5000,  
    min_df=2,           
    max_df=0.95,        
    ngram_range=(1, 3),  
    stop_words=russian_stopwords      
)

tfidf_matrix = vectorizer.fit_transform(df)
print(f"Размер TF-IDF матрицы: {tfidf_matrix.shape}")
print(f"Количество признаков: {len(vectorizer.get_feature_names_out())}")

Размер TF-IDF матрицы: (661, 3260)
Количество признаков: 3260


## Определение оптимального количества кластеров

In [21]:
from sklearn.decomposition import NMF

In [22]:
# создание модели NMF
nmf_model = NMF(n_components=5, random_state=0)
nmf_model.fit(tfidf_matrix)

# вывод топ слов для каждой темы
for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i}: {', '.join([vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

Topic 0: жизнь, вместе, лет, однажды, своей, однако, время, девушка, решает, становится
Topic 1: br, фильм, год, семья, попадает, схватку, рассказывает, мира, война, друзья
Topic 2: войны, мировой войны, мировой, второй мировой, второй мировой войны, второй, время второй мировой, время второй, время, немецкой
Topic 3: года, концерт, рамках, шоу, 2017, мирового, 2018, the, 2008 года, 2008
Topic 4: история, история жизни, жизни, br, хоккеиста, человека, российской, карьеры, молодого человека, молодого


In [23]:
nmf_model = NMF(n_components=7, random_state=0)
nmf_model.fit(tfidf_matrix)

for i, topic in enumerate(nmf_model.components_):
    print(f"Topic {i}: {', '.join([vectorizer.get_feature_names_out()[i] for i in topic.argsort()[:-11:-1]])}")

Topic 0: жизнь, однажды, своей, девушка, жизни, лет, становится, своего, решает, начинает
Topic 1: br, фильм, год, семья, попадает, рассказывает, схватку, война, фильм рассказывает, существует
Topic 2: войны, мировой войны, мировой, второй мировой, второй мировой войны, второй, время второй, время второй мировой, время, немецкой
Topic 3: года, концерт, рамках, шоу, 2017, 2018, мирового, the, 2008 года, 2008
Topic 4: история, история жизни, жизни, br, человека, хоккеиста, российской, карьеры, молодого человека, молодого
Topic 5: гарри, хогвартс, школы, волшебства хогвартс, курс школы чародейства, чародейства волшебства, чародейства, чародейства волшебства хогвартс, школы чародейства, курс школы
Topic 6: вместе, отправляется, путешествие, которые, мир, предстоит, отправляются, самые, друг, времени


In [30]:
X_topics_nmf = nmf_model.fit_transform(tfidf_matrix)
df['topic_nmf'] = np.argmax(X_topics_nmf, axis=1)

In [31]:
df.head(10)

0    Редакция Кинопоиска разбирается, как устроена ...
1    Тяжело переживая трагедию, из‑за которой ей пр...
2    В жизни Дениса Кукояки хватало съемок, приключ...
3    На Землю нападают пришельцы из космоса. Просто...
4    Семья поручила Минато Кусуноки присматривать з...
5    История Паши Техника — от яркого представителя...
6    Ночь Хэллоуина никогда не была ещё столь крова...
7    Великая Отечественная война. Чтобы спастись от...
8    Жизнь владельца убыточного видеопроката Михаэл...
9    Из-за квантового сбоя технологии и ИИ вышли из...
Name: description, dtype: object